In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSVs (files are in current working directory)
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')

# Ensure timestamp is integer
df_metric['timestamp'] = df_metric['timestamp'].astype(int)
df_trace['timestamp'] = df_trace['timestamp'].astype(int)

# Helper to convert epoch seconds to localized ISO string
def to_local_str(ts):
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

# --- metric.csv summaries ---
metric_rows = len(df_metric)

metric_unique_cmdb = pd.Series(sorted(df_metric['cmdb_id'].unique())).reset_index(drop=True).to_frame('cmdb_id').head(50)
metric_unique_kpis = pd.Series(sorted(df_metric['kpi_name'].unique())).reset_index(drop=True).to_frame('kpi_name').head(50)

metric_global_min_ts_int = int(df_metric['timestamp'].min()) if metric_rows>0 else None
metric_global_max_ts_int = int(df_metric['timestamp'].max()) if metric_rows>0 else None
metric_global_min_ts = (metric_global_min_ts_int, to_local_str(metric_global_min_ts_int)) if metric_rows>0 else (None, None)
metric_global_max_ts = (metric_global_max_ts_int, to_local_str(metric_global_max_ts_int)) if metric_rows>0 else (None, None)

# Aggregate grouped by (cmdb_id, kpi_name)
if metric_rows>0:
    g = df_metric.groupby(['cmdb_id','kpi_name'])
    metric_agg = g.agg(
        count=('value','count'),
        min_timestamp=('timestamp','min'),
        max_timestamp=('timestamp','max'),
        min_value=('value','min'),
        median_value=('value', 'median'),
        p95_value=('value', lambda x: float(np.percentile(x.dropna(),95)) if x.dropna().size>0 else np.nan)
    ).reset_index()
    # convert min/max timestamps to localized strings for readability (keep ints also)
    metric_agg['min_ts_local'] = metric_agg['min_timestamp'].apply(lambda x: to_local_str(int(x)))
    metric_agg['max_ts_local'] = metric_agg['max_timestamp'].apply(lambda x: to_local_str(int(x)))
    metric_agg_top20 = metric_agg.sort_values('count', ascending=False).head(20)[
        ['cmdb_id','kpi_name','count','min_timestamp','min_ts_local','max_timestamp','max_ts_local','min_value','median_value','p95_value']
    ]
else:
    metric_agg_top20 = pd.DataFrame(columns=['cmdb_id','kpi_name','count','min_timestamp','min_ts_local','max_timestamp','max_ts_local','min_value','median_value','p95_value'])

# --- trace.csv summaries ---
trace_rows = len(df_trace)

trace_unique_cmdb = pd.Series(sorted(df_trace['cmdb_id'].unique())).reset_index(drop=True).to_frame('cmdb_id').head(50)
trace_unique_names = pd.Series(sorted(df_trace['trace_name'].unique())).reset_index(drop=True).to_frame('trace_name').head(50)

trace_global_min_ts_int = int(df_trace['timestamp'].min()) if trace_rows>0 else None
trace_global_max_ts_int = int(df_trace['timestamp'].max()) if trace_rows>0 else None
trace_global_min_ts = (trace_global_min_ts_int, to_local_str(trace_global_min_ts_int)) if trace_rows>0 else (None, None)
trace_global_max_ts = (trace_global_max_ts_int, to_local_str(trace_global_max_ts_int)) if trace_rows>0 else (None, None)

# Aggregate grouped by (cmdb_id, trace_name)
if trace_rows>0:
    gt = df_trace.groupby(['cmdb_id','trace_name'])
    trace_agg = gt.agg(
        count=('value','count'),
        min_timestamp=('timestamp','min'),
        max_timestamp=('timestamp','max'),
        min_value=('value','min'),
        median_value=('value', 'median'),
        p95_value=('value', lambda x: float(np.percentile(x.dropna(),95)) if x.dropna().size>0 else np.nan)
    ).reset_index()
    trace_agg['min_ts_local'] = trace_agg['min_timestamp'].apply(lambda x: to_local_str(int(x)))
    trace_agg['max_ts_local'] = trace_agg['max_timestamp'].apply(lambda x: to_local_str(int(x)))
    trace_agg_top20 = trace_agg.sort_values('count', ascending=False).head(20)[
        ['cmdb_id','trace_name','count','min_timestamp','min_ts_local','max_timestamp','max_ts_local','min_value','median_value','p95_value']
    ]
else:
    trace_agg_top20 = pd.DataFrame(columns=['cmdb_id','trace_name','count','min_timestamp','min_ts_local','max_timestamp','max_ts_local','min_value','median_value','p95_value'])

# Final displayed variables (compact)
metric_rows, metric_unique_cmdb, metric_unique_kpis, metric_global_min_ts, metric_global_max_ts, metric_agg_top20, trace_rows, trace_unique_cmdb, trace_unique_names, trace_global_min_ts, trace_global_max_ts, trace_agg_top20

```
Out[1]:
```
```python
# Summarize results in plain English based on previously computed variables
metric_cmdb_count = df_metric['cmdb_id'].nunique()
metric_kpi_count = df_metric['kpi_name'].nunique()
trace_cmdb_count = df_trace['cmdb_id'].nunique()
trace_name_count = df_trace['trace_name'].nunique()

# Top metric group (highest count) from aggregated table
if not metric_agg_top20.empty:
    m0 = metric_agg_top20.iloc[0]
    metric_top_desc = f"Top metric group: ({m0.cmdb_id}, {m0.kpi_name}) with count={int(m0.count)}, time_range={m0.min_ts_local} to {m0.max_ts_local}, min={m0.min_value}, median={m0.median_value}, p95={m0.p95_value}"
else:
    metric_top_desc = "No metric groups available."

# Top trace group (highest count) from aggregated table
if not trace_agg_top20.empty:
    t0 = trace_agg_top20.iloc[0]
    trace_top_desc = f"Top trace group: ({t0.cmdb_id}, {t0.trace_name}) with count={int(t0.count)}, time_range={t0.min_ts_local} to {t0.max_ts_local}, min={t0.min_value}, median={t0.median_value}, p95={t0.p95_value}"
else:
    trace_top_desc = "No trace groups available."

summary = (
    "Summary of analysis:\n"
    f"- metric.csv: total rows = {metric_rows}; unique cmdb_id = {metric_cmdb_count}; unique kpi_name = {metric_kpi_count}.\n"
    f"  Global time range = {metric_global_min_ts[1]} to {metric_global_max_ts[1]} (UTC+8).\n"
    f"  {metric_top_desc}\n"
    "- Top 20 metric groups (by count) are available in `metric_agg_top20` (compact table).\n\n"
    f"- trace.csv: total rows = {trace_rows}; unique cmdb_id = {trace_cmdb_count}; unique trace_name = {trace_name_count}.\n"
    f"  Global time range = {trace_global_min_ts[1]} to {trace_global_max_ts[1]} (UTC+8).\n"
    f"  {trace_top_desc}\n"
    "- Top 20 trace groups (by count) are available in `trace_agg_top20` (compact table).\n\n"
    "Notes: All timestamps shown are converted to Asia/Shanghai (UTC+8)."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(90603,       cmdb_id
0      db_001
1      db_002
2      db_003
3      db_004
4      db_005
..        ...
45  redis_002
46  redis_003
47  redis_004
48  redis_005
49  redis_006

[50 rows x 1 columns],             kpi_name
0                ACS
1               AIOS
2                AWS
3         Agent_ping
4        Asm_Free_Tb
..               ...
45      Memory_total
46       Memory_used
47   Memory_used_pct
48   New_Tbs_Free_Gb
49  New_Tbs_Used_Pct

[50 rows x 1 columns], (1590863400, '2020-05-31 02:30:00 CST'), (1590867000, '2020-05-31 03:30:00 CST'),       cmdb_id                  kpi_name  count  min_timestamp             min_ts_local  max_timestamp             max_ts_local  min_value  median_value   p95_value
1779  osb_001              app.avg_time     61     1590863400  2020-05-31 02:30:00 CST     1590867000  2020-05-31 03:30:00 CST   0.000422      0.000477    0.022340
1780  osb_001                   app.num     61     1590863400  2020-05-31 02:30:00 CST     1590867000  2020-05-31 03:30:00 CST  15.000000    445.000000  513.000000
1781  osb_001            app.succee_num     61     1590863400  2020-05-31 02:30:00 CST     1590867000  2020-05-31 03:30:00 CST   9.500000    445.000000  513.000000
1759   os_022  Outgoing_network_traffic     61     1590863400  2020-05-31 02:30:00 CST     1590867000  2020-05-31 03:30:00 CST   0.006512      0.060601    0.069709
1254   os_012      Processor_load_5_min     61     1590863400  2020-05-31 02:30:00 CST     1590867000  2020-05-31 03:30:00 CST   0.005000      0.010000    0.021667
320    db_008              CPU_free_pct     61     1590863400  2020-05-31 02:30:00 CST     1590867000  2020-05-31 03:30:00 CST  96.090615     97.298075   97.876220
953    os_006                Send_total     61     1590863400  2020-05-31 02:30:00 CST     1590867000  2020-05-31 03:30:00 CST   0.000308      0.000368    0.000473
1393   os_015  Incoming_network_traffic     61     1590863400  2020-05-31 02:30:00 CST     1590867000  2020-05-31 03:30:00 CST   0.000000      0.002310    0.003416
1636   os_020              Disk_io_util     61     1590863400  2020-05-31 02:30:00 CST     1590867000  2020-05-31 03:30:00 CST   0.000000      0.000000    0.000000
95     db_003              CPU_free_pct     61     1590863400  2020-05-31 02:30:00 CST     1590867000  2020-05-31 03:30:00 CST  95.284303     98.828445   99.235046
1055   os_008                Send_total     61     1590863400  2020-05-31 02:30:00 CST     1590867000  2020-05-31 03:30:00 CST   0.000506      0.000565    0.000686
1782  osb_001           app.succee_rate     61     1590863400  2020-05-31 02:30:00 CST     1590867000  2020-05-31 03:30:00 CST   0.707300      1.000000    1.000000
974    os_007               Disk_rd_ios     61     1590863400  2020-05-31 02:30:00 CST     1590867000  2020-05-31 03:30:00 CST   0.000000      0.000000    0.000000
288    db_007             MEM_real_util     61     1590863400  2020-05-31 02:30:00 CST     1590867000  2020-05-31 03:30:00 CST  80.460000     80.480000   80.490000
1514   os_017                Send_total     61     1590863400  2020-05-31 02:30:00 CST     1590867000  2020-05-31 03:30:00 CST   0.018343      0.237143    0.270933
5      db_001              CPU_free_pct     61     1590863400  2020-05-31 02:30:00 CST     1590867000  2020-05-31 03:30:00 CST  96.875887     98.699331   98.975271
1228   os_012              Disk_io_util     61     1590863400  2020-05-31 02:30:00 CST     1590867000  2020-05-31 03:30:00 CST   0.000000      1.000000    2.000000
1473   os_017                Agent_ping     60     1590863400  2020-05-31 02:30:00 CST     1590866940  2020-05-31 03:29:00 CST   1.000000      1.000000    1.000000
1475   os_017              CPU_idle_pct     60     1590863400  2020-05-31 02:30:00 CST     1590866940  2020-05-31 03:29:00 CST  97.342324     97.740475   98.478121
1487   os_017               Disk_wr_ios     60     1590863400  2020-05-31 02:30:00 CST     1590866940  2020-05-31 03:29:00 CST   1.000000      3.000000    3.000000, 6232,       cmdb_id
0  docker_001
1  docker_002
2  docker_003
3  docker_004
4  docker_005
5  docker_006
6  docker_007
7  docker_008
8      os_021
9      os_022,                                            trace_name
0   trace.from_docker_001.remoteprocess.duration_mean
1    trace.from_docker_001.remoteprocess.duration_p95
2      trace.from_docker_001.remoteprocess.error_rate
3       trace.from_docker_001.remoteprocess.row_count
4   trace.from_docker_002.remoteprocess.duration_mean
..                                                ...
45     trace.to_docker_001.remoteprocess.duration_p95
46       trace.to_docker_001.remoteprocess.error_rate
47        trace.to_docker_001.remoteprocess.row_count
48    trace.to_docker_002.remoteprocess.duration_mean
49     trace.to_docker_002.remoteprocess.duration_p95

[50 rows x 1 columns], (1590865200, '2020-05-31 03:00:00 CST'), (1590866940, '2020-05-31 03:29:00 CST'),        cmdb_id                                     trace_name  count  min_timestamp             min_ts_local  max_timestamp             max_ts_local  min_value  median_value    p95_value
0   docker_001  trace.from_os_022.remoteprocess.duration_mean     30     1590865200  2020-05-31 03:00:00 CST     1590866940  2020-05-31 03:29:00 CST   0.321206      0.556859    74.299807
1   docker_001   trace.from_os_022.remoteprocess.duration_p95     30     1590865200  2020-05-31 03:00:00 CST     1590866940  2020-05-31 03:29:00 CST   0.894500      1.762200    91.690690
2   docker_001     trace.from_os_022.remoteprocess.error_rate     30     1590865200  2020-05-31 03:00:00 CST     1590866940  2020-05-31 03:29:00 CST   0.000000      0.000000     0.000000
3   docker_001      trace.from_os_022.remoteprocess.row_count     30     1590865200  2020-05-31 03:00:00 CST     1590866940  2020-05-31 03:29:00 CST   6.000000    107.000000   119.550000
4   docker_001                   trace.self.csf.duration_mean     30     1590865200  2020-05-31 03:00:00 CST     1590866940  2020-05-31 03:29:00 CST   0.039939      0.053567     4.285340
5   docker_001                    trace.self.csf.duration_p95     30     1590865200  2020-05-31 03:00:00 CST     1590866940  2020-05-31 03:29:00 CST   0.096000      0.176450     5.783805
6   docker_001                      trace.self.csf.error_rate     30     1590865200  2020-05-31 03:00:00 CST     1590866940  2020-05-31 03:29:00 CST   0.000000      0.000000     0.000000
7   docker_001                       trace.self.csf.row_count     30     1590865200  2020-05-31 03:00:00 CST     1590866940  2020-05-31 03:29:00 CST  21.000000    436.000000   478.200000
8   docker_001             trace.self.flyremote.duration_mean     30     1590865200  2020-05-31 03:00:00 CST     1590866940  2020-05-31 03:29:00 CST   0.006443      0.008620     2.006250
9   docker_001              trace.self.flyremote.duration_p95     30     1590865200  2020-05-31 03:00:00 CST     1590866940  2020-05-31 03:29:00 CST   0.007000      0.009700     2.012837
10  docker_001                trace.self.flyremote.error_rate     30     1590865200  2020-05-31 03:00:00 CST     1590866940  2020-05-31 03:29:00 CST   0.000000      0.000000     0.000000
11  docker_001                 trace.self.flyremote.row_count     30     1590865200  2020-05-31 03:00:00 CST     1590866940  2020-05-31 03:29:00 CST   6.000000    109.000000   119.550000
12  docker_001                  trace.self.jdbc.duration_mean     30     1590865200  2020-05-31 03:00:00 CST     1590866940  2020-05-31 03:29:00 CST   0.001659      0.002579     2.001025
13  docker_001                   trace.self.jdbc.duration_p95     30     1590865200  2020-05-31 03:00:00 CST     1590866940  2020-05-31 03:29:00 CST   0.003000      0.003000     2.010192
14  docker_001                     trace.self.jdbc.error_rate     30     1590865200  2020-05-31 03:00:00 CST     1590866940  2020-05-31 03:29:00 CST   0.000000      0.000000     0.000000
15  docker_001                      trace.self.jdbc.row_count     30     1590865200  2020-05-31 03:00:00 CST     1590866940  2020-05-31 03:29:00 CST  56.000000   1745.500000  1914.350000
16  docker_001                 trace.self.local.duration_mean     30     1590865200  2020-05-31 03:00:00 CST     1590866940  2020-05-31 03:29:00 CST   0.006845      0.012983     2.285824
17  docker_001                  trace.self.local.duration_p95     30     1590865200  2020-05-31 03:00:00 CST     1590866940  2020-05-31 03:29:00 CST   0.046950      0.089500     4.009828
18  docker_001                    trace.self.local.error_rate     30     1590865200  2020-05-31 03:00:00 CST     1590866940  2020-05-31 03:29:00 CST   0.000000      0.000000     0.000000
19  docker_001                     trace.self.local.row_count     30     1590865200  2020-05-31 03:00:00 CST     1590866940  2020-05-31 03:29:00 CST  50.000000   1526.000000  1673.700000)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

# Compute global P95 and P05 per (cmdb_id, kpi_name) using entire cached df_metric
import numpy as np
import pandas as pd
import pytz

# Reuse to_local_str if available; otherwise define it
try:
    to_local_str
except NameError:
    tz = pytz.timezone('Asia/Shanghai')
    def to_local_str(ts):
        return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

# 1) Calculate global thresholds (P95 and P05) using all data (must be done BEFORE filtering)
thr = (df_metric
       .groupby(['cmdb_id','kpi_name'])['value']
       .agg(
           global_P95=lambda x: float(np.percentile(x.dropna(),95)) if x.dropna().size>0 else np.nan,
           global_P05=lambda x: float(np.percentile(x.dropna(),5)) if x.dropna().size>0 else np.nan
       )
       .reset_index()
      )

# 2) Filter metric rows to the incident window [1590865200, 1590867000)
start_ts = 1590865200
end_ts = 1590867000
df_window = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] < end_ts)].copy()

# 3) Merge thresholds into windowed rows
df_win = df_window.merge(thr, on=['cmdb_id','kpi_name'], how='left')

# 4) Detect anomalies: high if value >= global_P95, low if value <= global_P05
df_win['is_high'] = df_win['value'] >= df_win['global_P95']
df_win['is_low'] = df_win['value'] <= df_win['global_P05']
df_win['is_anom'] = df_win['is_high'] | df_win['is_low']

# 5) Aggregate anomaly groups
df_anom_rows = df_win[df_win['is_anom']].copy()
total_anomalies = len(df_anom_rows)

if total_anomalies > 0:
    # direction and earliest timestamp per group from anomaly rows
    agg_anom = (df_anom_rows
                .groupby(['cmdb_id','kpi_name'])
                .agg(
                    anomaly_count=('is_anom','size'),
                    earliest_anom_ts=('timestamp','min'),
                    any_high=('is_high','any'),
                    any_low=('is_low','any')
                )
                .reset_index()
               )
    # Determine direction label
    def direction_label(row):
        if row['any_high'] and row['any_low']:
            return 'both'
        if row['any_high']:
            return 'high'
        return 'low'
    agg_anom['anomaly_direction'] = agg_anom.apply(direction_label, axis=1)
    # Convert earliest timestamp to local string
    agg_anom['earliest_anomaly_timestamp_local'] = agg_anom['earliest_anom_ts'].apply(lambda t: to_local_str(int(t)))
    # Drop helper cols
    agg_anom = agg_anom.drop(columns=['any_high','any_low','earliest_anom_ts'])
    
    # Get min/max value in the window for each group (from all window rows, not only anomalies)
    window_stats = (df_window
                    .groupby(['cmdb_id','kpi_name'])['value']
                    .agg(min_value_in_window='min', max_value_in_window='max')
                    .reset_index()
                   )
    # Merge global thresholds into aggregated anomalies
    agg_anom = (agg_anom
                .merge(window_stats, on=['cmdb_id','kpi_name'], how='left')
                .merge(thr, on=['cmdb_id','kpi_name'], how='left')
               )
    # Reorder and select requested columns
    agg_anom = agg_anom[[
        'cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp_local','anomaly_direction',
        'max_value_in_window','min_value_in_window','global_P95','global_P05'
    ]]
    # Sort by anomaly_count desc and limit to top 20
    anomalies_top20 = agg_anom.sort_values('anomaly_count', ascending=False).head(20).reset_index(drop=True)
else:
    # Empty result structure
    anomalies_top20 = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp_local','anomaly_direction',
        'max_value_in_window','min_value_in_window','global_P95','global_P05'
    ])

# Final compact outputs: anomalies_top20 and total_anomalies
anomalies_top20, total_anomalies

```
Out[2]:
```
```python
# Summarize anomaly detection results (using cached anomalies_top20 and total_anomalies)
import pandas as pd

# Use existing variables from kernel
try:
    anomalies_top20
    total_anomalies
except NameError:
    raise RuntimeError("Required variables 'anomalies_top20' and 'total_anomalies' not found in the kernel.")

n_reported_groups = anomalies_top20.shape[0]
earliest_anom_ts = anomalies_top20['earliest_anomaly_timestamp_local'].min() if n_reported_groups>0 else 'N/A'
unique_directions = sorted(anomalies_top20['anomaly_direction'].unique().tolist()) if n_reported_groups>0 else []

top5 = anomalies_top20[['cmdb_id','kpi_name','anomaly_count','anomaly_direction']].head(5)

summary = (
    f"Anomaly detection summary:\n"
    f"- Total anomalous points in the incident window: {int(total_anomalies)}.\n"
    f"- Number of (cmdb_id, kpi_name) groups with anomalies reported (top 20 shown): {n_reported_groups}.\n"
    f"- Earliest anomaly timestamp in reported groups (UTC+8): {earliest_anom_ts}.\n"
    f"- Anomaly direction(s) observed among reported groups: {', '.join(unique_directions) if unique_directions else 'N/A'}.\n"
    f"- Top group: ({top5.iloc[0].cmdb_id}, {top5.iloc[0].kpi_name}) with anomaly_count={int(top5.iloc[0].anomaly_count)} (direction={top5.iloc[0].anomaly_direction}).\n"
    f"- Top 5 anomaly groups (cmdb_id, kpi_name, anomaly_count, direction) are shown below."
)

summary, top5
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id              kpi_name  anomaly_count earliest_anomaly_timestamp_local anomaly_direction  max_value_in_window  min_value_in_window    global_P95    global_P05
0      db_001                  Hang             30          2020-05-31 03:00:00 CST              both         1.000000e+00         1.000000e+00  1.000000e+00  1.000000e+00
1   redis_012       used_memory_rss             30          2020-05-31 03:00:00 CST              both         2.544013e+09         2.544013e+09  2.544013e+09  2.544013e+09
2      db_001         MEM_real_util             30          2020-05-31 03:00:00 CST              both         4.625000e+01         4.617000e+01  4.618000e+01  4.617000e+01
3      db_001                  AIOS             30          2020-05-31 03:00:00 CST              both         1.000000e-02         0.000000e+00  1.000000e-02  0.000000e+00
4      db_001             MEM_Total             30          2020-05-31 03:00:00 CST              both         3.819023e+02         3.819023e+02  3.819023e+02  3.819023e+02
5   redis_012      used_memory_peak             30          2020-05-31 03:00:00 CST              both         3.006680e+09         3.006680e+09  3.006680e+09  3.006680e+09
6   redis_011       blocked_clients             30          2020-05-31 03:00:00 CST              both         0.000000e+00         0.000000e+00  0.000000e+00  0.000000e+00
7   redis_011     connected_clients             30          2020-05-31 03:00:00 CST              both         2.400000e+01         2.400000e+01  2.400000e+01  2.400000e+01
8   redis_011          evicted_keys             30          2020-05-31 03:00:00 CST              both         0.000000e+00         0.000000e+00  0.000000e+00  0.000000e+00
9   redis_011          expired_keys             30          2020-05-31 03:00:00 CST              both         0.000000e+00         0.000000e+00  0.000000e+00  0.000000e+00
10  redis_011         keyspace_hits             30          2020-05-31 03:00:00 CST              both         3.000000e+00         3.000000e+00  3.000000e+00  3.000000e+00
11  redis_010  rejected_connections             30          2020-05-31 03:00:00 CST              both         0.000000e+00         0.000000e+00  0.000000e+00  0.000000e+00
12  redis_010      used_memory_peak             30          2020-05-31 03:00:00 CST              both         2.920852e+07         2.920852e+07  2.920852e+07  2.920852e+07
13  redis_011       Redis_key_count             30          2020-05-31 03:00:00 CST              both         1.939700e+04         1.939700e+04  1.939700e+04  1.939700e+04
14  redis_010         keyspace_hits             30          2020-05-31 03:00:00 CST              both         3.000000e+00         3.000000e+00  3.000000e+00  3.000000e+00
15  redis_010       keyspace_misses             30          2020-05-31 03:00:00 CST              both         0.000000e+00         0.000000e+00  0.000000e+00  0.000000e+00
16  redis_010             maxmemory             30          2020-05-31 03:00:00 CST              both         2.000000e+09         2.000000e+09  2.000000e+09  2.000000e+09
17  redis_010            redis_load             30          2020-05-31 03:00:00 CST              both         0.000000e+00         0.000000e+00  0.000000e+00  0.000000e+00
18  redis_010            redis_ping             30          2020-05-31 03:00:00 CST              both         1.000000e+00         1.000000e+00  1.000000e+00  1.000000e+00
19  redis_009      used_memory_peak             30          2020-05-31 03:00:00 CST              both         3.625788e+07         3.625788e+07  3.625788e+07  3.625788e+07, 24799)```
```

In [ ]:
# In[4]:

import numpy as np
import pandas as pd

# Reuse cached variables: anomalies_top20, thr, df_window, df_metric (if present)
try:
    anomalies_top20
except NameError:
    raise RuntimeError("Expected 'anomalies_top20' in kernel. Please run previous steps.")

# 1) Filter out degenerate groups where global_P95 == global_P05 (use isclose for floats; require both finite)
gP95 = anomalies_top20['global_P95'].astype(float)
gP05 = anomalies_top20['global_P05'].astype(float)
degenerate_mask = np.isfinite(gP95) & np.isfinite(gP05) & np.isclose(gP95, gP05)
filtered_anom = anomalies_top20[~degenerate_mask].reset_index(drop=True)

# 1a) If filtering removed all rows (or user requested recompute), recompute anomalies excluding degenerate groups using cached full metric thresholds 'thr' and window 'df_window'
if filtered_anom.shape[0] == 0:
    # Ensure thresholds 'thr' and windowed rows 'df_window' exist
    try:
        thr
        df_window
    except NameError:
        raise RuntimeError("Required cached variables 'thr' and 'df_window' not found to recompute anomalies.")
    # exclude degenerate groups based on thr
    thr_finite = thr.copy()
    thr_finite['is_degenerate'] = thr_finite.apply(
        lambda r: np.isfinite(r['global_P95']) and np.isfinite(r['global_P05']) and np.isclose(r['global_P95'], r['global_P05']),
        axis=1
    )
    valid_groups = thr_finite[~thr_finite['is_degenerate']][['cmdb_id','kpi_name']]
    # merge and detect anomalies
    df_win2 = df_window.merge(thr, on=['cmdb_id','kpi_name'], how='left')
    df_win2 = df_win2.merge(valid_groups.assign(valid=1), on=['cmdb_id','kpi_name'], how='left')
    df_win2 = df_win2[df_win2['valid']==1].copy()
    df_win2['is_high'] = df_win2['value'] >= df_win2['global_P95']
    df_win2['is_low'] = df_win2['value'] <= df_win2['global_P05']
    df_win2['is_anom'] = df_win2['is_high'] | df_win2['is_low']
    df_anom_rows2 = df_win2[df_win2['is_anom']].copy()
    total_anom2 = len(df_anom_rows2)
    if total_anom2 > 0:
        agg2 = (df_anom_rows2.groupby(['cmdb_id','kpi_name'])
                .agg(anomaly_count=('is_anom','size'),
                     earliest_anom_ts=('timestamp','min'),
                     any_high=('is_high','any'),
                     any_low=('is_low','any'))
                .reset_index())
        agg2['anomaly_direction'] = agg2.apply(lambda r: 'both' if r['any_high'] and r['any_low'] else ('high' if r['any_high'] else 'low'), axis=1)
        agg2['earliest_anomaly_timestamp_local'] = agg2['earliest_anom_ts'].apply(lambda t: pd.to_datetime(int(t), unit='s', utc=True).tz_convert(pytz.timezone('Asia/Shanghai')).strftime('%Y-%m-%d %H:%M:%S %Z'))
        # window min/max values
        ws = df_window.groupby(['cmdb_id','kpi_name'])['value'].agg(min_value_in_window='min', max_value_in_window='max').reset_index()
        filtered_anom = (agg2.merge(ws, on=['cmdb_id','kpi_name'], how='left').merge(thr, on=['cmdb_id','kpi_name'], how='left'))
        filtered_anom = filtered_anom[[
            'cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp_local','anomaly_direction',
            'max_value_in_window','min_value_in_window','global_P95','global_P05'
        ]].sort_values('anomaly_count', ascending=False).reset_index(drop=True)
    else:
        # remain empty
        filtered_anom = pd.DataFrame(columns=[
            'cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp_local','anomaly_direction',
            'max_value_in_window','min_value_in_window','global_P95','global_P05'
        ])

# 2) Top 20 groups sorted by anomaly_count
top20_groups = filtered_anom.sort_values('anomaly_count', ascending=False).head(20).reset_index(drop=True)

# 3) Component-level summary by prefix: os_, docker_, db_, redis_, others
def prefix_of(cmdb):
    if isinstance(cmdb, str):
        for p in ('os_','docker_','db_','redis_'):
            if cmdb.startswith(p):
                return p.rstrip('_') + '_'  # keep format 'os_', 'docker_', etc.
    return 'others'

if top20_groups.shape[0] > 0:
    top20_groups['prefix'] = top20_groups['cmdb_id'].apply(prefix_of)
    comp_summary = (top20_groups
                    .groupby('prefix')
                    .agg(number_of_faulty_groups=('kpi_name','nunique'),
                         total_anomalous_points=('anomaly_count','sum'))
                    .reset_index()
                    .sort_values('total_anomalous_points', ascending=False)
                    .head(20)
                   )
else:
    comp_summary = pd.DataFrame(columns=['prefix','number_of_faulty_groups','total_anomalous_points'])

# 4) Candidate components (up to 10) from allowed sets with highest total_anomalous_points
# Build allowed sets
allowed_os = {f'os_{i:03d}' for i in range(1,23)}
allowed_docker = {f'docker_{i:03d}' for i in range(1,9)}
allowed_db = {f'db_{i:03d}' for i in range(1,14)}
allowed_set = allowed_os | allowed_docker | allowed_db

# Compute per-component totals from filtered_anom (use all groups in filtered_anom, not only top20)
all_filtered = filtered_anom.copy()
if all_filtered.shape[0] > 0:
    comp_totals = (all_filtered
                   .groupby('cmdb_id')
                   .agg(total_anomalous_points=('anomaly_count','sum'),
                        number_of_faulty_groups=('kpi_name','nunique'),
                        earliest_anomaly_timestamp_local=('earliest_anomaly_timestamp_local','min'))
                   .reset_index()
                  )
    # filter to allowed set
    candidates = comp_totals[comp_totals['cmdb_id'].isin(allowed_set)].sort_values('total_anomalous_points', ascending=False).head(10).reset_index(drop=True)
else:
    candidates = pd.DataFrame(columns=['cmdb_id','total_anomalous_points','number_of_faulty_groups','earliest_anomaly_timestamp_local'])

# Final compact outputs
top20_groups_selected = top20_groups[[
    'cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp_local','anomaly_direction',
    'max_value_in_window','min_value_in_window','global_P95','global_P05'
]]

top20_groups_selected, comp_summary, candidates

```
Out[4]:
```
```python
# Summarize final results in plain English using cached DataFrames
try:
    top20 = top20_groups_selected
    comp = comp_summary
    cand = candidates
except NameError:
    raise RuntimeError("Expected cached results 'top20_groups_selected', 'comp_summary', and 'candidates' in the kernel.")

# Build summary text
lines = []
lines.append("Anomaly summary after filtering degenerate (constant) series:")
if top20.shape[0] == 0:
    lines.append("- No non-degenerate anomaly groups found in the incident window.")
else:
    lines.append(f"- Number of reported non-degenerate anomaly groups (top shown): {top20.shape[0]}.")
    # List each group (up to 20)
    for _, r in top20.iterrows():
        lines.append(
            f"  - {r.cmdb_id} | {r.kpi_name}: anomalies={int(r.anomaly_count)}, "
            f"earliest={r.earliest_anomaly_timestamp_local}, direction={r.anomaly_direction}, "
            f"max_in_window={r.max_value_in_window}, min_in_window={r.min_value_in_window}, "
            f"P95={r.global_P95}, P05={r.global_P05}"
        )

# Component-level summary
if comp.shape[0] == 0:
    lines.append("- No component-level summary available.")
else:
    lines.append("\nComponent-level summary (by prefix):")
    for _, r in comp.iterrows():
        lines.append(f"  - {r.prefix}: faulty_groups={int(r.number_of_faulty_groups)}, total_anomalous_points={int(r.total_anomalous_points)}")

# Candidate components (allowed root-cause sets)
if cand.shape[0] == 0:
    lines.append("\nNo candidate components found in the allowed root-cause sets.")
else:
    lines.append("\nTop candidate components (allowed set) by anomalous points:")
    for _, r in cand.iterrows():
        lines.append(f"  - {r.cmdb_id}: total_anomalous_points={int(r.total_anomalous_points)}, faulty_groups={int(r.number_of_faulty_groups)}, earliest_anomaly={r.earliest_anomaly_timestamp_local}")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(  cmdb_id       kpi_name  anomaly_count earliest_anomaly_timestamp_local anomaly_direction  max_value_in_window  min_value_in_window  global_P95  global_P05
0  db_001  MEM_real_util             30          2020-05-31 03:00:00 CST              both                46.25                46.17       46.18       46.17
1  db_001           AIOS             30          2020-05-31 03:00:00 CST              both                 0.01                 0.00        0.01        0.00,   prefix  number_of_faulty_groups  total_anomalous_points
0    db_                        2                      60,   cmdb_id  total_anomalous_points  number_of_faulty_groups earliest_anomaly_timestamp_local
0  db_001                      60                        2          2020-05-31 03:00:00 CST)```
```

In [ ]:
# In[5]:

import numpy as np
import pandas as pd
import pytz

# Reuse cached variables: df_metric, df_trace, thr, df_window, start_ts, end_ts, to_local_str
# Ensure required vars exist
for v in ('df_metric','df_trace','thr','df_window','start_ts','end_ts'):
    if v not in globals():
        raise RuntimeError(f"Required variable '{v}' not found in kernel state.")

tz = pytz.timezone('Asia/Shanghai')
expected_window_len = int((end_ts - start_ts) / 60)  # expected samples in window (30)

# 1) KPI summary for cmdb_id = 'db_001'
cm = 'db_001'
df_cm_all = df_metric[df_metric['cmdb_id'] == cm].copy()
kpis = sorted(df_cm_all['kpi_name'].unique())[:50]  # limit <=50

rows = []
for k in kpis:
    # global thresholds from thr (precomputed on full series)
    thr_row = thr[(thr['cmdb_id'] == cm) & (thr['kpi_name'] == k)]
    if not thr_row.empty:
        global_P95 = float(thr_row['global_P95'].values[0])
        global_P05 = float(thr_row['global_P05'].values[0])
    else:
        global_P95 = np.nan
        global_P05 = np.nan
    # full series count
    count_total = int(df_cm_all[df_cm_all['kpi_name'] == k].shape[0])
    # windowed stats
    df_w = df_window[(df_window['cmdb_id'] == cm) & (df_window['kpi_name'] == k)].copy()
    count_in_window = int(df_w.shape[0])
    if count_in_window > 0:
        earliest_ts = int(df_w['timestamp'].min())
        latest_ts = int(df_w['timestamp'].max())
        earliest_local = pd.to_datetime(earliest_ts, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
        latest_local = pd.to_datetime(latest_ts, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
        min_val = float(df_w['value'].min())
        median_val = float(df_w['value'].median())
        max_val = float(df_w['value'].max())
        # breaches: any value >= global_P95 or <= global_P05 (use NaN-safe comparisons)
        breaches = False
        if np.isfinite(global_P95):
            breaches = breaches or (df_w['value'] >= global_P95).any()
        if np.isfinite(global_P05):
            breaches = breaches or (df_w['value'] <= global_P05).any()
    else:
        earliest_local = None
        latest_local = None
        min_val = np.nan
        median_val = np.nan
        max_val = np.nan
        breaches = False
    continuous_full_window = (count_in_window == expected_window_len)
    rows.append({
        'kpi_name': k,
        'global_P95': global_P95,
        'global_P05': global_P05,
        'count_total': count_total,
        'count_in_window': count_in_window,
        'earliest_timestamp_in_window_local': earliest_local,
        'latest_timestamp_in_window_local': latest_local,
        'min_value_in_window': min_val,
        'median_value_in_window': median_val,
        'max_value_in_window': max_val,
        'breaches_threshold': breaches,
        'continuous_full_window': continuous_full_window
    })

kpi_summary_df = pd.DataFrame(rows)
# Keep compact ordering: show KPIs with breaches first, then by count_in_window desc
kpi_summary_df = kpi_summary_df.sort_values(['breaches_threshold','count_in_window'], ascending=[False, False]).reset_index(drop=True)

# 2) Trace groups related to db_001: cmdb_id == 'db_001' OR trace_name contains 'db_001'
mask = (df_trace['cmdb_id'] == cm) | df_trace['trace_name'].str.contains(cm, na=False)
df_trace_sel = df_trace[mask].copy()

if not df_trace_sel.empty:
    def agg_p95(x): 
        xx = x.dropna()
        return float(np.percentile(xx,95)) if xx.size>0 else np.nan

    gt = (df_trace_sel
          .groupby(['cmdb_id','trace_name'])['value']
          .agg(count='count',
               min_timestamp='min',
               max_timestamp='max',
               min_value='min',
               median_value='median',
               p95_value=lambda x: agg_p95(x))
          .reset_index()
         )
    # localize timestamps
    gt['min_timestamp_local'] = gt['min_timestamp'].apply(lambda t: pd.to_datetime(int(t), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z'))
    gt['max_timestamp_local'] = gt['max_timestamp'].apply(lambda t: pd.to_datetime(int(t), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z'))

    # derive helper columns: error_rate_median and duration_p95 (if trace_name indicates)
    def error_rate_median_fn(row):
        if 'error_rate' in row['trace_name']:
            return row['median_value']
        return np.nan
    def duration_p95_fn(row):
        # if trace_name mentions duration_p95 or duration_mean or duration
        if ('duration_p95' in row['trace_name']) or ('duration.' in row['trace_name']) or ('duration' in row['trace_name']):
            return row['p95_value']
        return np.nan

    gt['error_rate_median'] = gt.apply(error_rate_median_fn, axis=1)
    gt['duration_p95'] = gt.apply(duration_p95_fn, axis=1)

    trace_agg_df = gt.sort_values('count', ascending=False).head(20).reset_index(drop=True)
else:
    trace_agg_df = pd.DataFrame(columns=['cmdb_id','trace_name','count','min_timestamp_local','max_timestamp_local','min_value','median_value','p95_value','error_rate_median','duration_p95'])

# 3) Single-line summary: earliest breach timestamp for any db_001 KPI in window and total anomalous points for db_001 in window
# Recompute anomalies for db_001 in window using global thresholds (thr)
thr_db = thr[thr['cmdb_id'] == cm].set_index('kpi_name')
df_w_db = df_window[df_window['cmdb_id'] == cm].copy()
if not df_w_db.empty:
    # merge thresholds
    df_w_db = df_w_db.merge(thr_db[['global_P95','global_P05']].reset_index(), on='kpi_name', how='left')
    df_w_db['is_high'] = df_w_db['value'] >= df_w_db['global_P95']
    df_w_db['is_low'] = df_w_db['value'] <= df_w_db['global_P05']
    df_w_db['is_anom'] = df_w_db['is_high'] | df_w_db['is_low']
    anom_rows_db = df_w_db[df_w_db['is_anom']].copy()
    total_anomalous_points_db = int(anom_rows_db.shape[0])
    if total_anomalous_points_db > 0:
        earliest_breach_ts = int(anom_rows_db['timestamp'].min())
        earliest_breach_local = pd.to_datetime(earliest_breach_ts, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    else:
        earliest_breach_local = None
else:
    total_anomalous_points_db = 0
    earliest_breach_local = None

single_line_summary = f"db_001 earliest_breach={earliest_breach_local}, total_anomalous_points_in_window={total_anomalous_points_db}"

# Final compact outputs
kpi_summary_df, trace_agg_df, single_line_summary

```
Out[5]:
```
Summary for component db_001 (plain English):

- KPIs examined: 45 distinct kpi_name entries for db_001 (results limited to ≤50).  
- Global thresholds: For each KPI we computed global P95 and P05 using the entire metric series.  
- Window coverage (incident window = [1590865200, 1590867000) => 2020-05-31 03:00:00 to 2020-05-31 03:30:00 CST):
  - Many KPIs have full coverage in the window: count_in_window = 30 (expected 30 samples) and continuous_full_window = True for the majority.
  - Some KPIs have sparse data in-window (e.g., single/two samples at 2020-05-31 03:27:00 CST).
- Breaches vs thresholds:
  - Every KPI row checked shows whether any in-window value exceeded the global_P95 or fell below global_P05. In the computed table many KPIs are marked breaches_threshold = True (i.e., at least one in-window point breached global P95 or P05).
  - Examples of KPIs with full-window breaches include: ACS, AIOS, AWS, CPU_free_pct, Call_Per_Sec (each had 30 in-window points and breached thresholds).
  - A few KPIs are degenerate/constant (global_P95 == global_P05 or single-value series); these still appear as breaches when their constant value equals the threshold — treat those cautiously.
- Counts and anomalies:
  - Earliest breach for any db_001 KPI in the window: 2020-05-31 03:00:00 CST (UTC+8).
  - Total anomalous points for db_001 inside the window: 592.
- Trace data:
  - No matching trace groups were found for db_001 (no rows where cmdb_id == 'db_001' or trace_name contains 'db_001' in the provided trace.csv).
- Interpretation notes:
  - Many db_001 KPIs show threshold breaches across the entire 30-minute window, suggesting sustained deviations rather than isolated spikes.  
  - Some KPI series are constant or have very few samples; treat those as degenerate when judging root cause.  
  - Next steps: inspect top-breached KPIs (those with highest anomaly_count and continuous_full_window=True) and correlate with application logs, traces from related services, and node/container metrics to localize root cause.

The original code execution output of IPython Kernel is also provided below for reference:

(          kpi_name    global_P95    global_P05  count_total  count_in_window earliest_timestamp_in_window_local latest_timestamp_in_window_local  min_value_in_window  median_value_in_window  max_value_in_window  breaches_threshold  continuous_full_window
0              ACS      0.020000      0.009500           60               30            2020-05-31 03:00:00 CST          2020-05-31 03:29:00 CST             0.000000                0.020000             0.020000                True                    True
1             AIOS      0.010000      0.000000           60               30            2020-05-31 03:00:00 CST          2020-05-31 03:29:00 CST             0.000000                0.000000             0.010000                True                    True
2              AWS      0.020000      0.000000           60               30            2020-05-31 03:00:00 CST          2020-05-31 03:29:00 CST             0.000000                0.010000             0.030000                True                    True
3     CPU_free_pct     98.975271     97.937506           61               30            2020-05-31 03:00:00 CST          2020-05-31 03:29:00 CST            96.875887               98.713052            98.973581                True                    True
4     Call_Per_Sec     11.400000     10.700000           60               30            2020-05-31 03:00:00 CST          2020-05-31 03:29:00 CST            10.600000               11.000000            18.400000                True                    True
..             ...           ...           ...          ...              ...                                ...                              ...                  ...                     ...                  ...                 ...                     ...
40     Tbs_Free_Gb    551.390000    551.390000            2                1            2020-05-31 03:27:00 CST          2020-05-31 03:27:00 CST           551.390000              551.390000           551.390000                True                   False
41    Tbs_Used_Pct     86.200000     86.200000            2                1            2020-05-31 03:27:00 CST          2020-05-31 03:27:00 CST            86.200000               86.200000            86.200000                True                   False
42  Total_Tbs_Size  11415.270000  11415.270000            1                1            2020-05-31 03:27:00 CST          2020-05-31 03:27:00 CST         11415.270000            11415.270000         11415.270000                True                   False
43     UndoTbs_Pct      2.150000      2.150000            2                1            2020-05-31 03:27:00 CST          2020-05-31 03:27:00 CST             2.150000                2.150000             2.150000                True                   False
44   Used_Tbs_Size   9149.690000   9149.690000            1                1            2020-05-31 03:27:00 CST          2020-05-31 03:27:00 CST          9149.690000             9149.690000          9149.690000                True                   False

[45 rows x 12 columns], Empty DataFrame
Columns: [cmdb_id, trace_name, count, min_timestamp_local, max_timestamp_local, min_value, median_value, p95_value, error_rate_median, duration_p95]
Index: [], 'db_001 earliest_breach=2020-05-31 03:00:00 CST, total_anomalous_points_in_window=592')

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import numpy as np
import pandas as pd
import pytz

# Reuse cached variables if available
for v in ('df_metric','thr','df_window','start_ts','end_ts'):
    if v not in globals():
        raise RuntimeError(f"Required variable '{v}' not found in kernel state.")

tz = pytz.timezone('Asia/Shanghai')
expected_window_len = int((end_ts - start_ts) / 60)  # should be 30

# helper to localize timestamp
def to_local_str_safe(ts):
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

cm = 'db_001'

# All KPIs for db_001 from full series
df_cm_all = df_metric[df_metric['cmdb_id'] == cm].copy()
kpi_names = sorted(df_cm_all['kpi_name'].unique())[:50]  # limit to 50 as requested

# thresholds for db_001 from precomputed 'thr'
thr_db = thr[thr['cmdb_id'] == cm].set_index('kpi_name')

rows = []
for k in kpi_names:
    # global thresholds (from thr computed on full series)
    if k in thr_db.index:
        global_P95 = float(thr_db.loc[k, 'global_P95'])
        global_P05 = float(thr_db.loc[k, 'global_P05'])
    else:
        global_P95 = np.nan
        global_P05 = np.nan

    # total count in full series
    count_total = int(df_cm_all[df_cm_all['kpi_name'] == k].shape[0])

    # windowed subset (pre-filtered df_window)
    df_w = df_window[(df_window['cmdb_id'] == cm) & (df_window['kpi_name'] == k)].copy()
    count_in_window = int(df_w.shape[0])

    if count_in_window > 0:
        earliest_ts = int(df_w['timestamp'].min())
        latest_ts = int(df_w['timestamp'].max())
        earliest_local = to_local_str_safe(earliest_ts)
        latest_local = to_local_str_safe(latest_ts)
        min_val = float(df_w['value'].min())
        median_val = float(df_w['value'].median())
        max_val = float(df_w['value'].max())
        # breaches: any in-window point >= global_P95 or <= global_P05 (NaN-safe)
        breach_high = False
        breach_low = False
        if np.isfinite(global_P95):
            breach_high = (df_w['value'] >= global_P95).any()
        if np.isfinite(global_P05):
            breach_low = (df_w['value'] <= global_P05).any()
        breaches = bool(breach_high or breach_low)
    else:
        earliest_local = None
        latest_local = None
        min_val = np.nan
        median_val = np.nan
        max_val = np.nan
        breaches = False

    continuous_full_window = (count_in_window == expected_window_len)

    # categorical boolean flags derived from kpi_name (case-insensitive)
    kn = k.lower()
    is_cpu = 'cpu' in kn
    is_mem = ('mem' in kn) or ('memory' in kn)
    is_conn = any(s in kn for s in ('conn','connection','rejected','blocked','clients'))
    is_net = any(s in kn for s in ('network','send','incoming','outgoing'))
    is_io = any(s in kn for s in ('io','disk','tbs'))
    is_hang = 'hang' in kn

    rows.append({
        'kpi_name': k,
        'global_P95': global_P95,
        'global_P05': global_P05,
        'count_total': count_total,
        'count_in_window': count_in_window,
        'earliest_timestamp_in_window_local': earliest_local,
        'latest_timestamp_in_window_local': latest_local,
        'min_value_in_window': min_val,
        'median_value_in_window': median_val,
        'max_value_in_window': max_val,
        'breaches_threshold': breaches,
        'continuous_full_window': continuous_full_window,
        'is_cpu': is_cpu,
        'is_mem': is_mem,
        'is_conn': is_conn,
        'is_net': is_net,
        'is_io': is_io,
        'is_hang': is_hang
    })

db_kpi_summary = pd.DataFrame(rows)

# Keep compact: sort so breached KPIs and those with more in-window points come first
db_kpi_summary = db_kpi_summary.sort_values(['breaches_threshold','count_in_window'], ascending=[False, False]).reset_index(drop=True)

# Limit rows to top 50 (as requested). There are <=50 by design.
db_kpi_summary_limited = db_kpi_summary.head(50)

# Connection-related KPI list (is_conn == True) with requested columns
conn_kpis = db_kpi_summary[db_kpi_summary['is_conn']].copy()
conn_kpi_list = conn_kpis[['kpi_name','count_in_window','breaches_threshold','earliest_timestamp_in_window_local']].reset_index(drop=True)

# Total number of in-window anomalous points across those connection-related KPIs
# Compute by counting rows in df_window for db_001 where kpi_name in conn_kpi_list and value breaches thresholds from thr_db
if conn_kpi_list.shape[0] > 0:
    conn_names = conn_kpi_list['kpi_name'].tolist()
    df_conn_win = df_window[(df_window['cmdb_id'] == cm) & (df_window['kpi_name'].isin(conn_names))].copy()
    # merge thresholds
    df_conn_win = df_conn_win.merge(thr_db[['global_P95','global_P05']].reset_index(), on='kpi_name', how='left')
    df_conn_win['is_anom'] = False
    if not df_conn_win.empty:
        # high and low checks with NaN-safe
        df_conn_win['is_high'] = np.where(np.isfinite(df_conn_win['global_P95']), df_conn_win['value'] >= df_conn_win['global_P95'], False)
        df_conn_win['is_low'] = np.where(np.isfinite(df_conn_win['global_P05']), df_conn_win['value'] <= df_conn_win['global_P05'], False)
        df_conn_win['is_anom'] = df_conn_win['is_high'] | df_conn_win['is_low']
    total_conn_anom_points = int(df_conn_win['is_anom'].sum())
else:
    total_conn_anom_points = 0

# Final compact outputs (use variables, not print)
db_kpi_summary_limited, conn_kpi_list, total_conn_anom_points

```
Out[6]:
```
```python
# Summarize db_001 KPI table and connection-related KPIs in plain English using cached results
try:
    db_tbl = db_kpi_summary_limited
    conn_list = conn_kpi_list
    conn_anom_pts = total_conn_anom_points
except NameError:
    raise RuntimeError("Expected cached variables 'db_kpi_summary_limited', 'conn_kpi_list', and 'total_conn_anom_points' in the kernel.")

# Compute compact stats
n_kpis = int(db_tbl.shape[0])
n_breached_kpis = int(db_tbl['breaches_threshold'].sum())
n_continuous = int(db_tbl['continuous_full_window'].sum())
# earliest breach among KPIs that breached (using available local timestamps)
breach_ts_candidates = db_tbl.loc[db_tbl['breaches_threshold'] & db_tbl['earliest_timestamp_in_window_local'].notna(), 'earliest_timestamp_in_window_local']
earliest_breach = breach_ts_candidates.min() if not breach_ts_candidates.empty else None
# connection-related list
n_conn_kpis = int(conn_list.shape[0])
conn_rows_sample = conn_list.head(10).to_dict('records')  # tiny sample for plain text

summary = (
    f"db_001 KPI summary (compact):\n"
    f"- Table produced with {n_kpis} KPI rows (variable: db_kpi_summary_limited). Each row includes global_P95/P05, counts, in-window stats, breach flag, continuity flag, and categorical flags.\n"
    f"- KPIs breaching thresholds in-window: {n_breached_kpis}.\n"
    f"- KPIs with full continuous coverage in the 30-minute window (30 samples): {n_continuous}.\n"
    f"- Earliest KPI breach observed (UTC+8): {earliest_breach}.\n\n"
    f"Connection-related KPIs (is_conn==True): {n_conn_kpis} rows. Example entries (kpi_name, count_in_window, breaches_threshold, earliest_timestamp_in_window_local):\n"
)

for r in conn_rows_sample:
    summary += f"  - {r['kpi_name']}, count_in_window={r['count_in_window']}, breaches={r['breaches_threshold']}, earliest={r['earliest_timestamp_in_window_local']}\n"

summary += f"\nTotal in-window anomalous points across these connection-related KPIs: {int(conn_anom_pts)}.\n\n"
summary += "Notes: Use the 'db_kpi_summary_limited' DataFrame for the full compact table (up to 50 rows). Connection KPIs and their aggregate anomalous-point count are in 'conn_kpi_list' and 'total_conn_anom_points' respectively."

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(          kpi_name    global_P95    global_P05  count_total  count_in_window earliest_timestamp_in_window_local latest_timestamp_in_window_local  min_value_in_window  median_value_in_window  max_value_in_window  breaches_threshold  continuous_full_window  is_cpu  is_mem  is_conn  is_net  is_io  is_hang
0              ACS      0.020000      0.009500           60               30            2020-05-31 03:00:00 CST          2020-05-31 03:29:00 CST             0.000000                0.020000             0.020000                True                    True   False   False    False   False  False    False
1             AIOS      0.010000      0.000000           60               30            2020-05-31 03:00:00 CST          2020-05-31 03:29:00 CST             0.000000                0.000000             0.010000                True                    True   False   False    False   False   True    False
2              AWS      0.020000      0.000000           60               30            2020-05-31 03:00:00 CST          2020-05-31 03:29:00 CST             0.000000                0.010000             0.030000                True                    True   False   False    False   False  False    False
3     CPU_free_pct     98.975271     97.937506           61               30            2020-05-31 03:00:00 CST          2020-05-31 03:29:00 CST            96.875887               98.713052            98.973581                True                    True    True   False    False   False  False    False
4     Call_Per_Sec     11.400000     10.700000           60               30            2020-05-31 03:00:00 CST          2020-05-31 03:29:00 CST            10.600000               11.000000            18.400000                True                    True   False   False    False   False  False    False
..             ...           ...           ...          ...              ...                                ...                              ...                  ...                     ...                  ...                 ...                     ...     ...     ...      ...     ...    ...      ...
40     Tbs_Free_Gb    551.390000    551.390000            2                1            2020-05-31 03:27:00 CST          2020-05-31 03:27:00 CST           551.390000              551.390000           551.390000                True                   False   False   False    False   False   True    False
41    Tbs_Used_Pct     86.200000     86.200000            2                1            2020-05-31 03:27:00 CST          2020-05-31 03:27:00 CST            86.200000               86.200000            86.200000                True                   False   False   False    False   False   True    False
42  Total_Tbs_Size  11415.270000  11415.270000            1                1            2020-05-31 03:27:00 CST          2020-05-31 03:27:00 CST         11415.270000            11415.270000         11415.270000                True                   False   False   False    False   False   True    False
43     UndoTbs_Pct      2.150000      2.150000            2                1            2020-05-31 03:27:00 CST          2020-05-31 03:27:00 CST             2.150000                2.150000             2.150000                True                   False   False   False    False   False   True    False
44   Used_Tbs_Size   9149.690000   9149.690000            1                1            2020-05-31 03:27:00 CST          2020-05-31 03:27:00 CST          9149.690000             9149.690000          9149.690000                True                   False   False   False    False   False   True    False

[45 rows x 18 columns],        kpi_name  count_in_window  breaches_threshold earliest_timestamp_in_window_local
0  Sess_Connect               30                True            2020-05-31 03:00:00 CST, 16)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```